# GroupVisualise: exploring the Cryosphere Community Datapool with `ccdtools`

**Hackathon: Choose Your Own Adventure. Group: Visualise**

Your group will use [`ccdtools`](https://ccdtools.readthedocs.io/latest/), the Python interface to the ACCESS-NRI
**Cryosphere Community Datapool (CCD)**, to find, load and interrogate Antarctic datasets. Then you'll make a
publication-quality figure.

There are four questions. Each has:
* 💡 **Hints**: click to expand. They point you at the right tools but don't give the answer.
* 🧩 **A code skeleton** with `...` placeholders for you to fill in.
* ✍️ **An answer cell** where you record what you found and *how* you know it's right.

> **Before you start**
> * This notebook must run on **NCI Gadi**, e.g. an ARE JupyterLab session. The datapool lives in `/g/data/av17`,
>   so you need to have **joined project `av17`**, and your ARE session needs `gdata/av17` in its *Storage* field.
> * You need an environment with `ccdtools` installed (`pip install access-ccdtools` or
>   `conda install accessnri::access_ccdtools`), plus `geopandas`, `xarray`, `dask` and `matplotlib`.
> * Some datasets are **big**. BedMachine is about 4 GB and ITS_LIVE Antarctica at 240 m is about 25 GB. They load
>   *lazily* (dask), so think before you call `.values`, `.compute()` or `.plot()` on a full-resolution array.

Useful links: [ccdtools docs](https://ccdtools.readthedocs.io/latest/) ·
[overview tutorial](https://ccdtools.readthedocs.io/latest/tutorials/1_CCD_Overview.html) ·
[API reference](https://ccdtools.readthedocs.io/latest/ccdtools.html) ·
[GitHub](https://github.com/ACCESS-NRI/ccdtools)

## 0. Set up

Run these cells as they are. If creating the catalog fails with a `PermissionError` or `FileNotFoundError`, check
that you're on Gadi, a member of `av17`, and that your ARE session mounts `gdata/av17`.

In [ ]:
import ccdtools
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

print("ccdtools version:", getattr(ccdtools, "__version__", "unknown"))

# Optional: a dask client makes large reductions faster and shows progress.
# from dask.distributed import Client
# client = Client(threads_per_worker=1)
# client

In [ ]:
# Create the catalog and show every dataset it knows about
catalog = ccdtools.catalog.DataCatalog()
catalog

### Your toolbox

These are the `DataCatalog` methods you'll need. Check the docstrings with `help(catalog.load_dataset)` and so on.

| Method | What it does |
|---|---|
| `catalog.datasets` | the catalog as a plain `pandas.DataFrame` (columns include `dataset`, `display_name`, `description`, `tags`, version, subdataset, …) |
| `catalog.search(keyword)` | filter by keyword (or a **list** of keywords) in the name, display name or tags. Returns a new, smaller catalog |
| `catalog.help(dataset, version=None)` | describe a dataset and the options it supports, **without loading it** |
| `catalog.available_versions(dataset)` | which versions exist |
| `catalog.available_subdatasets(dataset, version=None)` | e.g. coastline / basins / ice shelves |
| `catalog.available_resolutions(dataset, version=None, subdataset=None)` | for multi-resolution datasets (e.g. ITS_LIVE) |
| `catalog.load_dataset(dataset, version=None, subdataset=None, **kwargs)` | load it. Returns an `xarray.Dataset`, a `GeoDataFrame` or a `DataFrame`, depending on the file type |

In [ ]:
# Warm-up: look at the raw table and its columns
catalog.datasets.columns

---
## Question 1: which datasets include velocity data?

**What are the names of the datasets that include *any* velocity data?**

Give the `dataset` names, i.e. the strings you'd pass to `load_dataset`, not the display names.

<details><summary>💡 Hints (click to expand)</summary>

1. `catalog.search()` matches the keyword against the **dataset name, display name and tags**. Try more than one
   keyword, and remember `search` also accepts a **list**.
2. The catalog has one row per dataset × version × subdataset, so the same name can appear many times. How do
   you get each name only once from `catalog.datasets`?
3. **Be sceptical of tags.** A tag shows someone *thought* the dataset was related to velocity. It doesn't
   prove the files contain velocity fields. Is every match really velocity data? Some datasets may contain velocity
   without saying so in their name. How could you check a candidate cheaply? (`help()`? Loading a coarse
   subset and looking at `data_vars`?)
4. Do any datasets contain velocity-like quantities that *aren't* ice-flow velocity (e.g. rates of change)? Decide
   whether they count, and say so in your answer.
</details>

In [ ]:
# Q1: your code here

# 1. search the catalog with one or more keywords
velocity_catalog = ...   # TODO

# 2. get the unique dataset names from the result
velocity_names = ...     # TODO

# 3. (optional but recommended) check one or two candidates really hold velocity fields
...

✍️ **Q1 answer:**

* Dataset names:
* How we confirmed them (and anything we excluded, and why):

---
## Question 2: BedMachine Antarctica bed extremes

**What are the minimum and maximum bed topography values in the *latest* version of BedMachine Antarctica?
Which sectors of Antarctica are these points in?**

Use the **IMBIE basins** to name the sectors.

<details><summary>💡 Hints (click to expand)</summary>

**Finding and loading the data**
1. BedMachine's dataset name contains `bedmachine`. Use the catalog to find it, then find its **latest version**
   (don't assume it's `v1`). Which method lists versions?
2. The overview tutorial loads BedMachine and shows the variables (`bed`, `surface`, `thickness`, `mask`, …). The
   arrays are **lazy dask arrays**. `.min()` and `.max()` only build a task graph, so something has to trigger the
   computation.

**The extremes and where they are**
3. Getting the *value* is not enough. You need the **location** (x, y). Look at `xarray`'s `argmin`/`argmax`
   (with `dim=[...]`) or `idxmin`/`idxmax`. With 2D arrays, think about how to turn an index back into coordinates.
4. **Think about what "bed" means.** BedMachine provides bed elevation everywhere, including under the open ocean
   (bathymetry). Do you want the extremes over the **whole grid**, or only where there's **ice** (or grounded
   ice)? The `mask` variable's attributes (`flag_values`, `flag_meanings`) tell you what each mask value means.
   Report both if you're not sure, and say which is which.
5. Check the `units` and the vertical reference of `bed` in its attributes (is it relative to the geoid?).

**Naming the sector**
6. The IMBIE basins are a **subdataset** of the Antarctic boundaries dataset. Which method lists subdatasets? The
   result is a `GeoDataFrame`. Look at its columns (the tutorial shows `NAME`, `Regions`, `Subregions`, …).
7. To find which basin a point is in: make a point geometry (`gpd.points_from_xy`) and do a **spatial join**
   (`gpd.sjoin`), or test `basins.contains(point)`.
8. ⚠️ **Coordinate reference systems.** BedMachine x/y are in polar stereographic metres (EPSG:3031). Check
   `basins.crs` before joining. If they differ, reproject one of them.
9. A point under the ocean might fall **outside every basin polygon**. If your join returns nothing, what would be
   a sensible way to name the nearest sector? (`gpd.sjoin_nearest`?)
</details>

In [ ]:
# Q2: your code here

# 1. find BedMachine and its latest version
bm_name = ...             # TODO
bm_latest = ...           # TODO
bm = catalog.load_dataset(bm_name, version=bm_latest)

# 2. decide your domain (whole grid? ice only?) and compute the extremes AND their locations
bed = ...                 # TODO
bed_min, bed_max = ..., ...                 # TODO
xy_min, xy_max = ..., ...                   # TODO: (x, y) of each extreme

# 3. load the IMBIE basins and find which basin/region each point lies in
basins = ...              # TODO
...

✍️ **Q2 answer:**

| | Bed elevation (m) | x (m) | y (m) | IMBIE basin `NAME` | Region |
|---|---|---|---|---|---|
| Minimum | | | | | |
| Maximum | | | | | |

* BedMachine version used:
* Domain used (whole grid / ice only / grounded only):
* Does the location make physical sense? (e.g. a known trench or mountain range)

---
## Question 3: fastest and slowest ice shelves

**Which ice shelf has the highest velocity? Which has the lowest?**

<details><summary>💡 Hints (click to expand)</summary>

**Ice shelf outlines**
1. Ice shelf polygons are another **subdataset** of the Antarctic boundaries dataset. List the subdatasets and look
   for one about ice shelves.
2. Look at the `GeoDataFrame` columns. Is there a column that separates floating ice shelves from other polygons
   (look at the unique values of something like `TYPE`)? Is there a name column? Do some shelves appear as
   **several polygons**? Consider `dissolve(by=...)`.

**Velocity**
3. Choose one velocity dataset from your Q1 answer and justify it (coverage of the ice shelves, resolution,
   epoch). Use `available_resolutions` and `help` before loading. ITS_LIVE takes `subdataset`, `resolution`
   and `static` keywords.
4. If the dataset only has `vx` and `vy`, you need the **speed**.
5. Full-resolution Antarctic velocity is tens of GB. **Coarsen** it (`.coarsen(...).mean()`) to something
   manageable before doing per-shelf statistics. How much does the choice of coarsening change your answer?

**Per-shelf statistics (zonal statistics)**
6. You need one number per shelf. Some approaches you could research:
   * `rioxarray`'s `.rio.clip(geometries, crs)` for each shelf,
   * `regionmask` to build a labelled mask from the polygons,
   * `rasterio.features.rasterize` to burn the polygons onto the velocity grid, then a `groupby`.
7. **Define "velocity of an ice shelf".** Mean? Median? Maximum? A high percentile? Each gives a different
   answer, especially for the "highest". State your choice and why.
8. Watch out for:
   * **tiny shelves** with only a few pixels (should there be a minimum area or pixel count?),
   * **NaNs / no-data** areas in the velocity product,
   * **CRS** mismatches between the polygons and the velocity grid.
</details>

In [ ]:
# Q3: your code here

# 1. load the ice-shelf polygons and keep only floating ice shelves
shelves = ...             # TODO

# 2. load a velocity product (choose and justify!) and get the speed on a coarsened grid
vel = ...                 # TODO
speed = ...               # TODO

# 3. compute one statistic per ice shelf
shelf_stats = ...         # TODO: e.g. a DataFrame with name, area, n_pixels, mean/median/max speed

# 4. rank them
...

✍️ **Q3 answer:**

* Velocity dataset, version, resolution and coarsening used:
* Statistic used (mean / median / max / percentile) and why:
* Minimum shelf size applied (if any):
* **Highest-velocity ice shelf:**
* **Lowest-velocity ice shelf:**
* How sensitive is the ranking to your choices?

---
## Question 4: a publication-quality figure

**Make a plot of a 2D variable of your choice that would be suitable for a publication. Overlay the IMBIE
basins.**

Ideas: BedMachine bed or ice thickness, surface speed (log scale), geothermal heat flow, RACMO SMB, ice-shelf
height change, …

<details><summary>💡 Hints: what makes a figure "publication quality"? (click to expand)</summary>

**Data and projection**
1. Work in **EPSG:3031** (polar stereographic). Use `ax.set_aspect("equal")` so Antarctica isn't stretched.
2. **Coarsen** before plotting. A figure doesn't need 500 m pixels for the whole continent, and the render
   will be much faster.

**Colour**
3. Use a **perceptually uniform** colormap (`viridis`, `cividis`, or `cmocean` maps like `cmocean.cm.topo`,
   `deep` or `speed`). Avoid `jet`/`rainbow`. Check the figure works for colour-blind readers and in greyscale.
4. Use a **diverging** colormap centred on zero for signed quantities (bed relative to sea level, height change).
   Look at `matplotlib.colors.TwoSlopeNorm`. Use a **log** norm (`LogNorm`) for velocity.
5. Every colorbar needs a **label with units**.

**Overlay and annotation**
6. Plot the basins with `facecolor="none"` and thin edges. Consider a different line style or colour for the
   region boundaries (East / West / Peninsula). You could `dissolve(by="Regions")` the basins.
7. Label the basins or regions with text at a sensible point inside each polygon (`representative_point()` is
   always inside, `centroid` may not be).
8. Replace the axis ticks in metres with **km**, or remove the axes and add a **scale bar**.

**Output**
9. Size the figure for the journal: single column is about 8.5–9 cm wide, double column about 17–18 cm.
   `figsize` is in inches.
10. Font sizes should be legible at print size (about 7–9 pt). Save as **vector** (`.pdf`/`.svg`) or as a
    high-dpi `.png` (`dpi=300`) with `bbox_inches="tight"`.
11. Include a **data citation** (dataset name, version, DOI) in the caption. `catalog.help(...)` and
    `catalog.datasets` will help you find these.
</details>

In [ ]:
# Q4: your code here

# 1. choose and load your variable (coarsen it!)
field = ...               # TODO

# 2. load the IMBIE basins (reuse from Q2) and check the CRS matches
...

# 3. build the figure
fig, ax = plt.subplots(figsize=(..., ...))     # TODO: journal-sized
...                                             # TODO: field, colormap, norm, colorbar with units
...                                             # TODO: basin overlay, labels, km axes or scale bar

# 4. save it
# fig.savefig("groupvisualise_figure.pdf", bbox_inches="tight")
# fig.savefig("groupvisualise_figure.png", dpi=300, bbox_inches="tight")

✍️ **Q4: figure caption** (write it as you would for a paper, including data sources, versions and DOIs):

> **Figure 1.** …

---
## Wrap-up: report back to the hackathon

1. One-line answers to Q1–Q3.
2. Your figure from Q4.
3. **One thing that surprised you** about the data or about `ccdtools`.
4. **One thing `ccdtools` could do better.** If it's a bug or a feature request, consider opening an issue on
   [GitHub](https://github.com/ACCESS-NRI/ccdtools/issues).